# SUPERDIFF Composition Analysis - Interactive Notebook

This notebook provides an interactive interface for running and analyzing SUPERDIFF composition experiments.

## Quick Start

1. Configure your experiment parameters below
2. Run all cells (Runtime → Run all)
3. Explore the visualizations and diagnostics

## Table of Contents

1. [Setup & Configuration](#setup)
2. [Run Experiments](#experiments)
3. [Visual Analysis](#visual)
4. [Statistical Analysis](#stats)
5. [Manifold Geometry](#manifold)
6. [Interactive Exploration](#interactive)

## 1. Setup & Configuration <a name="setup"></a>

In [ ]:
# Imports
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from notebooks.composition_experiments import (
    ExperimentConfig, 
    CompositionExperimentSuite
)
from notebooks.manifold_geometry_analysis import (
    ManifoldGeometryAnalyzer,
    analyze_composition_geometry,
    analyze_trajectory_curvature
)

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports loaded successfully")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

### Configure Experiment Parameters

In [ ]:
# EXPERIMENT CONFIGURATION
# ========================

# Prompts to test
PROMPT_A = "A photograph of a cat"
PROMPT_B = "A photograph of a dog"
PROMPT_COMPOSED = "A photograph of a cat and a dog"  # Leave as None to auto-generate

# Sampling parameters
NUM_RUNS = 10          # Number of stochastic runs (10-20 recommended)
BATCH_SIZE = 4         # Samples per run
NUM_STEPS = 500        # Inference steps (500 for quality, 100 for speed)
GUIDANCE_SCALE = 7.5   # CFG scale
LIFT = 0.0            # SUPERDIFF lift parameter

# Output
OUTPUT_DIR = "experiments/interactive_analysis"

# Quick test mode (faster, lower quality)
QUICK_MODE = False

if QUICK_MODE:
    NUM_RUNS = 5
    NUM_STEPS = 100
    print("⚡ Quick mode enabled: Using reduced parameters")

# Create config
config = ExperimentConfig(
    prompt_a=PROMPT_A,
    prompt_b=PROMPT_B,
    prompt_composed=PROMPT_COMPOSED or f"{PROMPT_A} and {PROMPT_B}",
    num_runs=NUM_RUNS,
    batch_size=BATCH_SIZE,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    lift=LIFT,
    output_dir=OUTPUT_DIR
)

print("\n" + "="*60)
print("EXPERIMENT CONFIGURATION")
print("="*60)
print(f"Prompt A:        {config.prompt_a}")
print(f"Prompt B:        {config.prompt_b}")
print(f"Monolithic:      {config.prompt_composed}")
print(f"Runs:            {config.num_runs}")
print(f"Batch size:      {config.batch_size}")
print(f"Steps:           {config.num_inference_steps}")
print(f"Total samples:   {config.num_runs * config.batch_size}")
print(f"Output:          {config.output_dir}")
print("="*60)

## 2. Run Experiments <a name="experiments"></a>

This will take some time depending on your parameters:
- Quick mode (~10-15 min)
- Full quality (~30-45 min)

Progress will be displayed below.

In [ ]:
# Initialize experiment suite
suite = CompositionExperimentSuite(config)

# Run all experiments
suite.run_all_experiments()

print("\n✓ Experiments complete!")
print(f"✓ Results saved to: {suite.output_dir}")

## 3. Visual Analysis <a name="visual"></a>

Let's examine the generated images.

In [ ]:
# Display sample images comparison
from IPython.display import Image, display

display(Image(filename=str(suite.output_dir / 'sample_images_comparison.png')))

### Visual Inspection Questions

1. **Does SUPERDIFF produce hybrids or co-presence?**
   - Hybrids: Mixed features (cat-dog chimera)
   - Co-presence: Both objects visible

2. **How similar is SUPERDIFF to the monolithic prompt?**
   - Very similar → Mathematical AND ≈ linguistic "and"
   - Very different → Semantic mismatch

3. **Are outputs consistent across runs?**
   - Consistent → Low variance, stable composition
   - Variable → High variance, unstable composition

## 4. Statistical Analysis <a name="stats"></a>

### Centroid Distances

In [ ]:
# Compute centroids
latents_mono = torch.cat([l.cpu().flatten(1) for l in suite.results['monolithic']['latents']], dim=0)
latents_a = torch.cat([l.cpu().flatten(1) for l in suite.results['prompt_a']['latents']], dim=0)
latents_b = torch.cat([l.cpu().flatten(1) for l in suite.results['prompt_b']['latents']], dim=0)
latents_sd = torch.cat([l.cpu().flatten(1) for l in suite.results['superdiff']['latents']], dim=0)

centroid_mono = latents_mono.mean(dim=0)
centroid_a = latents_a.mean(dim=0)
centroid_b = latents_b.mean(dim=0)
centroid_sd = latents_sd.mean(dim=0)
centroid_midpoint = (centroid_a + centroid_b) / 2

# Compute key distances
dist_sd_midpoint = torch.norm(centroid_sd - centroid_midpoint).item()
dist_sd_mono = torch.norm(centroid_sd - centroid_mono).item()
dist_a_b = torch.norm(centroid_a - centroid_b).item()

# Normalized metric
normalized_distance = dist_sd_midpoint / dist_a_b

print("="*60)
print("CENTROID DISTANCE ANALYSIS")
print("="*60)
print(f"\nDistance from SUPERDIFF to:")
print(f"  (A+B)/2 (midpoint):  {dist_sd_midpoint:10.4f}")
print(f"  Monolithic prompt:   {dist_sd_mono:10.4f}")
print(f"\nReference:")
print(f"  Distance A to B:     {dist_a_b:10.4f}")
print(f"\nNormalized distance: {normalized_distance:10.4f}")
print(f"  (SUPERDIFF to midpoint, relative to A-B distance)")
print("\n" + "="*60)

# Interpretation
print("\nINTERPRETATION:")
if normalized_distance < 0.3:
    print("  ✓ SUPERDIFF performs approximate linear interpolation")
    print("    → Hybridization is likely INTRINSIC to the formula")
elif normalized_distance < 0.5:
    print("  ~ SUPERDIFF shows moderate deviation from linear interpolation")
    print("    → Some non-linear effects present")
else:
    print("  ✗ SUPERDIFF significantly deviates from linear interpolation")
    print("    → Strong non-linear composition")

if dist_sd_mono < dist_sd_midpoint:
    print("  ✓ SUPERDIFF is CLOSER to monolithic prompt")
    print("    → Mathematical AND aligns with linguistic semantics")
else:
    print("  ✗ SUPERDIFF is FARTHER from monolithic prompt")
    print("    → Mathematical AND differs from linguistic semantics")

In [ ]:
# Display centroid statistics plot
display(Image(filename=str(suite.output_dir / 'centroid_statistics.png')))

### Kappa Dynamics

In [ ]:
# Analyze kappa
kappas = torch.stack([k.cpu() for k in suite.results['superdiff']['kappas']], dim=0)
kappa_mean = kappas.mean().item()
kappa_std = kappas.std().item()

print("="*60)
print("KAPPA DYNAMICS ANALYSIS")
print("="*60)
print(f"\nMean κ:  {kappa_mean:8.4f}")
print(f"Std κ:   {kappa_std:8.4f}")
print(f"\nInterpretation:")
print(f"  κ = 0.5: Equal weighting between A and B")
print(f"  κ > 0.5: Biased toward A ('{config.prompt_a}')")
print(f"  κ < 0.5: Biased toward B ('{config.prompt_b}')")
print("\n" + "="*60)

if abs(kappa_mean - 0.5) < 0.1:
    print("  ✓ Balanced composition (κ ≈ 0.5)")
elif kappa_mean > 0.5:
    print(f"  ⚠ A-biased composition (κ = {kappa_mean:.3f})")
else:
    print(f"  ⚠ B-biased composition (κ = {kappa_mean:.3f})")

# Display kappa dynamics plot
display(Image(filename=str(suite.output_dir / 'kappa_dynamics.png')))

## 5. Manifold Geometry Analysis <a name="manifold"></a>

Advanced analysis of latent space geometry.

In [ ]:
# Run manifold geometry analysis
analyze_composition_geometry(
    latents_mono, 
    latents_a, 
    latents_b, 
    latents_sd,
    output_dir=str(suite.output_dir)
)

In [ ]:
# Display manifold geometry results
display(Image(filename=str(suite.output_dir / 'manifold_geometry_analysis.png')))

In [ ]:
# Display detailed results
results_file = suite.output_dir / 'manifold_geometry_results.txt'
if results_file.exists():
    with open(results_file, 'r') as f:
        print(f.read())

### Geodesic vs Euclidean Distance Interpretation

The **geodesic/Euclidean ratio** is a key diagnostic:

- **Ratio ≈ 1.0**: Latent paths follow manifold structure (on-manifold)
  - SUPERDIFF composition stays within learned distribution
  - Hybridization is inherent to the composition formula

- **Ratio > 1.2**: Paths take shortcuts through off-manifold regions
  - SUPERDIFF creates unrealistic intermediate states
  - Hybridization may be a geometric artifact
  - Potential for improvement with manifold-aware methods

## 6. Interactive Exploration <a name="interactive"></a>

Explore specific aspects of the results interactively.

In [ ]:
# Interactive plot: Select which run to visualize
from ipywidgets import interact, IntSlider
from IPython.display import display

@interact(run_idx=IntSlider(min=0, max=config.num_runs-1, step=1, value=0))
def show_run_images(run_idx):
    """Display images from a specific run"""
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    conditions = [
        ('monolithic', 'Monolithic'),
        ('prompt_a', 'Prompt A'),
        ('prompt_b', 'Prompt B'),
        ('superdiff', 'SUPERDIFF')
    ]
    
    for ax, (condition, label) in zip(axes, conditions):
        latents = suite.results[condition]['latents'][run_idx][0:1]
        from notebooks.utils import get_image
        img = get_image(suite.vae, latents, nrow=1, ncol=1)
        
        ax.imshow(img)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive plot: Trajectory distances over time
@interact(run_idx=IntSlider(min=0, max=config.num_runs-1, step=1, value=0))
def plot_trajectory_distances(run_idx):
    """Plot how trajectories diverge over time"""
    def flatten_trajectory(traj):
        return traj.trajectories.reshape(traj.trajectories.shape[0],
                                        traj.trajectories.shape[1], -1)
    
    traj_mono = flatten_trajectory(suite.results['monolithic']['trajectories'][run_idx])
    traj_a = flatten_trajectory(suite.results['prompt_a']['trajectories'][run_idx])
    traj_b = flatten_trajectory(suite.results['prompt_b']['trajectories'][run_idx])
    traj_sd = flatten_trajectory(suite.results['superdiff']['trajectories'][run_idx])
    
    # Compute distances (average over batch)
    dist_sd_mono = torch.norm(traj_sd - traj_mono, dim=2).mean(dim=1)
    dist_sd_a = torch.norm(traj_sd - traj_a, dim=2).mean(dim=1)
    dist_sd_b = torch.norm(traj_sd - traj_b, dim=2).mean(dim=1)
    
    plt.figure(figsize=(12, 6))
    plt.plot(dist_sd_mono.numpy(), label='SUPERDIFF vs Monolithic', 
             color='purple', linewidth=2)
    plt.plot(dist_sd_a.numpy(), label='SUPERDIFF vs A', 
             color='blue', linewidth=2)
    plt.plot(dist_sd_b.numpy(), label='SUPERDIFF vs B', 
             color='orange', linewidth=2)
    
    plt.xlabel('Diffusion Step', fontsize=12)
    plt.ylabel('L2 Distance', fontsize=12)
    plt.title(f'Trajectory Distances (Run {run_idx+1})', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Summary & Conclusions

Review the key findings from your analysis.

In [ ]:
# Display summary report
summary_file = suite.output_dir / 'summary_report.txt'
if summary_file.exists():
    with open(summary_file, 'r') as f:
        print(f.read())

## Export Results

Save key metrics for further analysis or publication.

In [ ]:
# Export to CSV
import pandas as pd

results_dict = {
    'prompt_a': config.prompt_a,
    'prompt_b': config.prompt_b,
    'dist_sd_midpoint': dist_sd_midpoint,
    'dist_sd_mono': dist_sd_mono,
    'dist_a_b': dist_a_b,
    'normalized_distance': normalized_distance,
    'kappa_mean': kappa_mean,
    'kappa_std': kappa_std,
    'num_runs': config.num_runs,
    'num_steps': config.num_inference_steps
}

df = pd.DataFrame([results_dict])
output_csv = suite.output_dir / 'results_summary.csv'
df.to_csv(output_csv, index=False)

print(f"✓ Results exported to: {output_csv}")
print("\nSummary:")
display(df.T)

## Next Steps

Based on your findings:

1. **If hybridization is due to linear interpolation**:
   - Try different composition operators
   - Investigate spatial conditioning methods
   - Test with prompts that have clear spatial semantics

2. **If hybridization is due to off-manifold trajectories**:
   - Implement geodesic interpolation
   - Try projection onto tangent spaces
   - Investigate manifold-aware composition

3. **If results match monolithic prompts**:
   - Composition successfully captures linguistic semantics
   - Test with more complex compositions
   - Explore multi-prompt composition (A ∧ B ∧ C)

4. **Parameter sensitivity**:
   - Sweep lift parameter
   - Vary guidance scale
   - Test with different model versions